# Dataset Gold MOPEP por consenso

Este notebook combina três tabelas silver: GPT-4o, Qwen e GPT-5 nano. Para cada exemplo, uma tag entra na tabela gold quando aparece em pelo menos dois dos três modelos.

O `example_id` existente é usado como identificador comum. A tabela privada relaciona esse identificador ao site. Antes de salvar os splits compartilháveis, a coluna `domain` é omitida. O texto de `business_model` não recebe outra limpeza.

## 1. Imports e configuração

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "analytics" / "src").is_dir()
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analytics.src.evaluate_mopep_tags import ALLOWED_TAGS

DATASETS_DIR = Path("/tmp/mopep-bmc-tags")
GPT5_ROOT = REPO_ROOT / "analytics/results/openai-silver"
QWEN_ROOT = REPO_ROOT / "analytics/results/mopep-silver/qwen"
PRIVATE_OUTPUT_ROOT = Path("/tmp/mopep-bmc-private")
SHAREABLE_OUTPUT_ROOT = (
    REPO_ROOT / "analytics/results/mopep-golden-shareable"
)

SPLITS = ["train", "calibration", "test"]

BASE_PATHS = {
    split: DATASETS_DIR / f"{split}.csv"
    for split in SPLITS
}

GPT5_PATHS = {
    split: GPT5_ROOT / f"{split}.csv"
    for split in SPLITS
}

## 2. Localizar as campanhas Qwen completas

Cada execução do Qwen cria um diretório com fingerprint. Para cada split, selecionamos a campanha completa que contém o mesmo número de exemplos da tabela-base.

In [ ]:
def qwen_predictions_path(split):
    expected_rows = len(
        pd.read_csv(
            BASE_PATHS[split],
            usecols=["example_id"],
        )
    )

    candidates = []

    for run_path in QWEN_ROOT.glob(
        f"{split}__qwen-silver__*/run.json"
    ):
        run = json.loads(
            run_path.read_text(encoding="utf-8")
        )

        if (
            run["status"] == "completed"
            and run["example_count"] == expected_rows
        ):
            candidates.append(
                (
                    run["updated_at"],
                    run_path.parent / "predictions.csv",
                )
            )

    return sorted(candidates)[-1][1]


QWEN_PATHS = {
    split: qwen_predictions_path(split)
    for split in SPLITS
}

pd.DataFrame(
    {
        "split": SPLITS,
        "base": [BASE_PATHS[split] for split in SPLITS],
        "gpt5_nano": [GPT5_PATHS[split] for split in SPLITS],
        "qwen": [QWEN_PATHS[split] for split in SPLITS],
    }
)

## 3. Carregar as predições

A tabela-base contém o voto do GPT-4o na coluna `tags`. As tabelas do Qwen e GPT-5 nano contêm seus votos na coluna `generated_tags`.

In [ ]:
def load_predictions(path, tags_column):
    predictions = pd.read_csv(
        path,
        usecols=["example_id", "generated_tags"],
    )

    predictions[tags_column] = (
        predictions["generated_tags"].apply(json.loads)
    )

    return predictions[["example_id", tags_column]]

## 4. Aplicar a regra de dois entre três

Cada modelo concede no máximo um voto por tag. A ordem final segue a taxonomia usada pelo projeto.

In [ ]:
VOTE_COLUMNS = [
    "gpt4o_tags",
    "qwen_tags",
    "gpt5_nano_tags",
]


def select_golden_tags(row):
    model_tag_sets = [
        set(row[column])
        for column in VOTE_COLUMNS
    ]

    return [
        tag
        for tag in ALLOWED_TAGS
        if sum(
            tag in model_tags
            for model_tags in model_tag_sets
        ) >= 2
    ]

## 5. Construir os splits gold

In [ ]:
def build_golden_split(split):
    base = pd.read_csv(BASE_PATHS[split])

    base = base.rename(
        columns={"tags": "gpt4o_tags"}
    )
    base["gpt4o_tags"] = (
        base["gpt4o_tags"].apply(json.loads)
    )

    qwen = load_predictions(
        QWEN_PATHS[split],
        "qwen_tags",
    )
    gpt5 = load_predictions(
        GPT5_PATHS[split],
        "gpt5_nano_tags",
    )

    golden = (
        base
        .merge(qwen, on="example_id", how="inner")
        .merge(gpt5, on="example_id", how="inner")
    )

    golden["golden_tags"] = golden.apply(
        select_golden_tags,
        axis=1,
    )
    golden["tags"] = golden["golden_tags"]
    golden["n_tags"] = golden["tags"].str.len()
    golden["label_source"] = "majority_vote_2_of_3"

    return golden

In [ ]:
golden_splits = {
    split: build_golden_split(split)
    for split in SPLITS
}

pd.DataFrame(
    [
        {"split": split, "rows": len(frame)}
        for split, frame in golden_splits.items()
    ]
)

## 6. Salvar a tabela privada de identidade

Esta tabela mantém a ligação entre o identificador e o site. Ela fica fora do diretório compartilhável.

In [ ]:
identity_table = pd.concat(
    [
        frame[["example_id", "domain", "split"]]
        for frame in golden_splits.values()
    ],
    ignore_index=True,
)

identity_table = identity_table.rename(
    columns={
        "example_id": "id",
        "domain": "site",
    }
)

PRIVATE_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

identity_path = PRIVATE_OUTPUT_ROOT / "identity.csv"
identity_table.to_csv(
    identity_path,
    index=False,
    encoding="utf-8",
)

identity_path

## 7. Salvar os splits compartilháveis

A coluna `domain` é removida pela seleção final de colunas. Cada CSV compartilhável contém somente `id`, `business_model` e `tags`.

In [ ]:
SHAREABLE_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

output_rows = []

for split, golden in golden_splits.items():
    shareable = golden[
        ["example_id", "business_model", "golden_tags"]
    ].copy()

    shareable = shareable.rename(
        columns={
            "example_id": "id",
            "golden_tags": "tags",
        }
    )
    shareable["tags"] = shareable["tags"].apply(
        lambda values: json.dumps(
            values,
            ensure_ascii=False,
        )
    )

    output_path = SHAREABLE_OUTPUT_ROOT / f"{split}.csv"
    shareable.to_csv(
        output_path,
        index=False,
        encoding="utf-8",
    )

    output_rows.append(
        {
            "split": split,
            "rows": len(shareable),
            "output_path": output_path,
        }
    )

pd.DataFrame(output_rows)

## Saídas

Tabela privada:

- `/tmp/mopep-bmc-private/identity.csv`

Tabelas compartilháveis:

- `analytics/results/mopep-golden-shareable/train.csv`
- `analytics/results/mopep-golden-shareable/calibration.csv`
- `analytics/results/mopep-golden-shareable/test.csv`